# Notebook 03 · Exploratory Data Analysis
**Input :** `F:\mmd\data\cleaned\*`  
**Output:** `F:\mmd\outputs\figures\*.png` + in-notebook charts  

---
### Mục tiêu EDA
| # | Câu hỏi | Ảnh hưởng đến model |
|---|---|---|
| 1 | Phân phối rating như thế nào? | Có dùng implicit hay explicit feedback? |
| 2 | Long-tail item/user? | Quyết định `min_count` threshold |
| 3 | Session length distribution? | Quyết định `max_seq_len` cho GRU4Rec |
| 4 | Temporal patterns? | Quyết định split point hợp lý |
| 5 | Cold-start ratio? | Đánh giá khó khăn của bài toán |
| 6 | Category distribution? | Feature engineering cho side-info |
| 7 | Price distribution? | Bin strategy cho price embedding |

## 0 · Imports & config

In [1]:
import gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pyarrow.dataset as ds
import psutil, pickle
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Plot style ────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGSIZE  = (12, 5)
COLOR_PRIMARY   = "#4C72B0"
COLOR_SECONDARY = "#DD8452"

# ── Paths ─────────────────────────────────────────────────────────────
ROOT_DIR    = Path(r"F:\amazon_data")
CLEANED_DIR = ROOT_DIR / "data" / "processed"
FIGURES_DIR = ROOT_DIR / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name): 
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved → {FIGURES_DIR/name}.png")

def ram(): 
    v = psutil.virtual_memory()
    return f"RAM {v.used/1e9:.1f}/{v.total/1e9:.1f} GB ({v.percent:.0f}%)"

print(f"Figures dir: {FIGURES_DIR}")
print(ram())

Figures dir: F:\amazon_data\outputs\figures
RAM 5.4/8.4 GB (63%)


## 1 · Load cleaned data

In [ ]:
# Reviews
df = pd.read_parquet(CLEANED_DIR / "review_clean.parquet")
print(f"Reviews : {len(df):,} rows  |  {df.columns.tolist()}")

# Meta
df_meta = pd.read_parquet(CLEANED_DIR / "meta_clean.parquet")
print(f"Meta    : {len(df_meta):,} rows  |  {df_meta.columns.tolist()}")

# Sessions
with open(CLEANED_DIR / "sessions.pkl", "rb") as f:
    sessions = pickle.load(f)
print(f"Sessions: {len(sessions):,} users")

print("\nReviews dtypes:")
print(df.dtypes)
print(ram())

## 2 · Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# ── 2a: Count per rating ──────────────────────────────────────────────
rating_vc = df["rating"].value_counts().sort_index()
axes[0].bar(rating_vc.index.astype(str), rating_vc.values, color=COLOR_PRIMARY, edgecolor="white")
axes[0].set_title("Rating Distribution (count)")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
for i, (r, c) in enumerate(rating_vc.items()):
    axes[0].text(i, c + c*0.01, f"{c/len(df)*100:.1f}%", ha="center", fontsize=9)

# ── 2b: Avg rating per item ───────────────────────────────────────────
item_avg = df.groupby("item_idx")["rating"].mean()
axes[1].hist(item_avg, bins=40, color=COLOR_SECONDARY, edgecolor="white")
axes[1].set_title("Item Average Rating Distribution")
axes[1].set_xlabel("Average Rating")
axes[1].set_ylabel("# Items")
axes[1].axvline(item_avg.mean(), color="red", ls="--", lw=1.5, label=f"mean={item_avg.mean():.2f}")
axes[1].legend()

plt.suptitle("Rating Analysis", fontweight="bold")
savefig("01_rating_distribution")

print(f"Overall mean rating : {df['rating'].mean():.3f}")
print(f"% 5-star reviews    : {(df['rating']==5).mean()*100:.1f}%")
print(f"% 1-star reviews    : {(df['rating']==1).mean()*100:.1f}%")

## 3 · User & Item Activity (Long-tail)

In [ ]:
user_counts = df["user_idx"].value_counts()
item_counts = df["item_idx"].value_counts()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ── User activity histogram ───────────────────────────────────────────
axes[0,0].hist(user_counts.values, bins=50, log=True, color=COLOR_PRIMARY, edgecolor="white")
axes[0,0].set_title("User Activity (log scale)")
axes[0,0].set_xlabel("# Reviews per User"); axes[0,0].set_ylabel("# Users (log)")

# ── Item popularity histogram ─────────────────────────────────────────
axes[0,1].hist(item_counts.values, bins=50, log=True, color=COLOR_SECONDARY, edgecolor="white")
axes[0,1].set_title("Item Popularity (log scale)")
axes[0,1].set_xlabel("# Reviews per Item"); axes[0,1].set_ylabel("# Items (log)")

# ── User CDF ─────────────────────────────────────────────────────────
sorted_u = np.sort(user_counts.values)
cdf_u    = np.arange(1, len(sorted_u)+1) / len(sorted_u)
axes[1,0].plot(sorted_u, cdf_u, color=COLOR_PRIMARY, lw=2)
axes[1,0].set_title("User Activity CDF")
axes[1,0].set_xlabel("# Reviews per User"); axes[1,0].set_ylabel("Cumulative fraction")
axes[1,0].axvline(np.percentile(sorted_u, 80), color="red", ls="--",
                   label=f"p80={np.percentile(sorted_u, 80):.0f}")
axes[1,0].legend()

# ── Item CDF ──────────────────────────────────────────────────────────
sorted_i = np.sort(item_counts.values)
cdf_i    = np.arange(1, len(sorted_i)+1) / len(sorted_i)
axes[1,1].plot(sorted_i, cdf_i, color=COLOR_SECONDARY, lw=2)
axes[1,1].set_title("Item Popularity CDF")
axes[1,1].set_xlabel("# Reviews per Item"); axes[1,1].set_ylabel("Cumulative fraction")
axes[1,1].axvline(np.percentile(sorted_i, 80), color="red", ls="--",
                   label=f"p80={np.percentile(sorted_i, 80):.0f}")
axes[1,1].legend()

plt.suptitle("User & Item Activity — Long Tail", fontweight="bold")
savefig("02_long_tail")

# Summary
print("User activity stats:")
print(user_counts.describe().to_string())
print("\nItem popularity stats:")
print(item_counts.describe().to_string())

# Long-tail: top 20% items cover how much of interactions?
top20_items = item_counts.nlargest(int(len(item_counts)*0.2))
coverage = top20_items.sum() / len(df)
print(f"\nTop 20% popular items cover {coverage*100:.1f}% of all interactions (long-tail: {coverage:.2f})")

## 4 · Session Length Distribution

In [ ]:
seq_lens = sessions["seq_len"]

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Histogram
axes[0].hist(seq_lens, bins=50, color=COLOR_PRIMARY, edgecolor="white")
axes[0].set_title("Session Length Distribution")
axes[0].set_xlabel("Session Length"); axes[0].set_ylabel("# Users")
for p, lbl in [(50, "p50"), (90, "p90"), (99, "p99")]:
    val = np.percentile(seq_lens, p)
    axes[0].axvline(val, ls="--", lw=1.5, label=f"{lbl}={val:.0f}")
axes[0].legend()

# Boxplot by bucket
buckets = pd.cut(seq_lens, bins=[0, 5, 10, 20, 50, 100, 200, 10000],
                 labels=["2-5","6-10","11-20","21-50","51-100","101-200","200+"])
bucket_counts = buckets.value_counts().sort_index()
axes[1].bar(bucket_counts.index.astype(str), bucket_counts.values,
            color=COLOR_SECONDARY, edgecolor="white")
axes[1].set_title("Session Length Buckets")
axes[1].set_xlabel("Length Bucket"); axes[1].set_ylabel("# Users")
for i, v in enumerate(bucket_counts.values):
    axes[1].text(i, v + v*0.01, f"{v/len(seq_lens)*100:.1f}%", ha="center", fontsize=9)

plt.suptitle("Session Length Analysis", fontweight="bold")
savefig("03_session_length")

print("Session length statistics:")
print(seq_lens.describe().to_string())
print(f"\n→ Recommended max_seq_len for GRU4Rec: {int(np.percentile(seq_lens, 95))} (p95)")

## 5 · Temporal Patterns

In [ ]:
TS_UNIT = "ms"   # đồng bộ với notebook 02
df["dt"] = pd.to_datetime(df["timestamp"], unit=TS_UNIT)
df["year_month"] = df["dt"].dt.to_period("M")

monthly = df.groupby("year_month").size().reset_index(name="count")
monthly["year_month_str"] = monthly["year_month"].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ── Monthly volume ────────────────────────────────────────────────────
axes[0].plot(monthly["year_month_str"], monthly["count"],
             color=COLOR_PRIMARY, lw=1.5, marker="o", ms=2)
axes[0].set_title("Monthly Review Volume")
axes[0].set_ylabel("# Reviews")
axes[0].tick_params(axis="x", rotation=45)
# Hiển thị mỗi 12 tháng để không rối
step = max(1, len(monthly)//20)
axes[0].set_xticks(monthly["year_month_str"][::step])

# ── Yearly total ──────────────────────────────────────────────────────
yearly = df.groupby(df["dt"].dt.year).size()
axes[1].bar(yearly.index.astype(str), yearly.values,
            color=COLOR_SECONDARY, edgecolor="white")
axes[1].set_title("Yearly Review Volume")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("# Reviews")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

plt.suptitle("Temporal Patterns", fontweight="bold")
savefig("04_temporal_patterns")

# Growth rate
yoy = yearly.pct_change().dropna() * 100
print("Year-over-year growth:")
for yr, pct in yoy.items():
    print(f"  {yr}: {pct:+.1f}%")

## 6 · Cold-start Analysis

In [ ]:
# Định nghĩa cold-start: user/item chỉ xuất hiện trong test set
# Sau LOO split: train_seq là những item trừ 2 cuối

# Items trong train
train_items = set()
for seq in sessions["train_seq"]:
    train_items.update(seq)

# Test items
test_items_all  = sessions["test_item"].values
cold_test_items = [i for i in test_items_all if i not in train_items]

cold_start_ratio = len(cold_test_items) / len(test_items_all)
print(f"Total test samples       : {len(test_items_all):,}")
print(f"Cold-start test items    : {len(cold_test_items):,}")
print(f"Cold-start ratio         : {cold_start_ratio*100:.2f}%")

# Phân tích theo frequency bin
item_freq_train = pd.Series(list(train_items)).value_counts()  # thực ra đây là tần suất xuất hiện
# Đếm frequency đúng hơn:
from collections import Counter
train_freq = Counter()
for seq in sessions["train_seq"]:
    train_freq.update(seq)

freq_bins = [1, 5, 10, 20, 50, 100, float("inf")]
bin_labels = ["1","2-5","6-10","11-20","21-50","51-100","100+"]
bin_counts = [0] * len(bin_labels)

for item in train_items:
    f = train_freq[item]
    for bi, (lo, hi) in enumerate(zip([0]+freq_bins[:-1], freq_bins)):
        if lo < f <= hi:
            bin_counts[bi] += 1
            break

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(bin_labels, bin_counts, color=COLOR_PRIMARY, edgecolor="white")
ax.set_title("Item Training Frequency Distribution")
ax.set_xlabel("# Training appearances"); ax.set_ylabel("# Items")
savefig("05_item_frequency_bins")

## 7 · Price Distribution

In [ ]:
price_valid = df_meta["price_usd"].dropna()
price_valid = price_valid[(price_valid > 0) & (price_valid < price_valid.quantile(0.99))]

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

axes[0].hist(price_valid, bins=60, color=COLOR_PRIMARY, edgecolor="white")
axes[0].set_title("Price Distribution (< p99)")
axes[0].set_xlabel("Price (USD)"); axes[0].set_ylabel("# Items")
axes[0].axvline(price_valid.median(), color="red", ls="--", lw=1.5,
                label=f"median=${price_valid.median():.2f}")
axes[0].legend()

axes[1].hist(np.log1p(price_valid), bins=60, color=COLOR_SECONDARY, edgecolor="white")
axes[1].set_title("Price Distribution (log scale)")
axes[1].set_xlabel("log(1 + price)"); axes[1].set_ylabel("# Items")

plt.suptitle("Price Analysis", fontweight="bold")
savefig("06_price_distribution")

# Price bins cho feature engineering
price_bins  = [0, 10, 25, 50, 100, 200, float("inf")]
price_lbls  = ["<$10", "$10-25", "$25-50", "$50-100", "$100-200", ">$200"]
df_meta["price_bin"] = pd.cut(df_meta["price_usd"], bins=price_bins, labels=price_lbls)
print("Price bin distribution:")
print(df_meta["price_bin"].value_counts().sort_index().to_string())
print(f"\nMissing price: {df_meta['price_usd'].isna().sum():,} "
      f"({df_meta['price_usd'].isna().mean()*100:.1f}%)")

## 8 · EDA Summary & Recommendations

In [ ]:
n_users = sessions["user_idx"].nunique()
n_items = df["item_idx"].nunique()
n_inter = len(df)

p50_seq = int(np.percentile(sessions["seq_len"], 50))
p95_seq = int(np.percentile(sessions["seq_len"], 95))

print("="*60)
print("  EDA SUMMARY")
print("="*60)
print(f"  Users             : {n_users:>10,}")
print(f"  Items             : {n_items:>10,}")
print(f"  Interactions      : {n_inter:>10,}")
print(f"  Sparsity          : {1-n_inter/(n_users*n_items):.6f}")
print(f"  Avg seq len (p50) : {p50_seq:>10}")
print(f"  Avg seq len (p95) : {p95_seq:>10}")
print(f"  Cold-start ratio  : {cold_start_ratio*100:>9.2f}%")
print("="*60)
print("\n  RECOMMENDATIONS FOR MODELING")
print("-"*60)
print(f"  Item2Vec window_size : 5 (typical for co-purchase)")
print(f"  Item2Vec embedding   : 128")
print(f"  GRU4Rec max_seq_len  : {p95_seq} (p95 of session length)")
print(f"  GRU4Rec hidden_size  : 256")
print(f"  Negative sampling    : popularity-based (do to long-tail)")
print("="*60)